In [1]:
# %%
from pathlib import Path
import json
import pickle

import numpy as np
import pandas as pd
from tqdm import tqdm

import faiss
from sentence_transformers import SentenceTransformer

c:\Users\aarav\anaconda3\envs\pmc_graphrag\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# %%
PROJECT_ROOT = Path("..").resolve()
PARSED_DIR = PROJECT_ROOT / "data" / "parsed"
INDEX_DIR = PROJECT_ROOT / "data" / "index"

INDEX_DIR.mkdir(parents=True, exist_ok=True)

IN_PARQUET = PARSED_DIR / "pmc_retrieval_candidates.parquet"

OUT_FAISS = INDEX_DIR / "chunks.faiss"
OUT_LOOKUP = INDEX_DIR / "chunk_lookup.parquet"
OUT_META = INDEX_DIR / "chunk_meta.json"

assert IN_PARQUET.exists(), "Missing retrieval candidates parquet"

In [3]:
# %%
df = pd.read_parquet(IN_PARQUET)

print("Chunks:", len(df))
print("PMCIDs:", df["pmcid"].nunique())

df.head(3)

Chunks: 12935
PMCIDs: 814


,chunk_id,pmcid,article_title,section,chunk_index,chunk_text,token_count,license
0,9fd0d269d90d3d89b02c81308d3a1500b6b91c37,PMC12446057,The immune checkpoint TIM-3/HMGB-1 axis in myo...,discussion,0,"we investigated, for the first time, expressio...",350,CC BY
1,79b3541338632b7d1dd88f1b98e362958d8a7411,PMC12446057,The immune checkpoint TIM-3/HMGB-1 axis in myo...,discussion,1,"an mi. on pbmc level, overall tim - 3 rna expr...",350,CC BY
2,5020b51f114f3fc6edf8746b2206e74869294b06,PMC12446057,The immune checkpoint TIM-3/HMGB-1 axis in myo...,discussion,2,"to be more organ - bound, possibly to maintain...",350,CC BY


In [4]:
# %%
EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

embedder = SentenceTransformer(EMBED_MODEL)
EMBED_DIM = embedder.get_sentence_embedding_dimension()

print("Embedding dim:", EMBED_DIM)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 682.60it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding dim: 384


In [5]:
# %%
texts = df["chunk_text"].tolist()

embeddings = []

BATCH = 64

for i in tqdm(range(0, len(texts), BATCH)):
    batch = texts[i:i+BATCH]
    vecs = embedder.encode(
        batch,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    )
    embeddings.append(vecs)

X = np.vstack(embeddings).astype("float32")

print("Embedding matrix:", X.shape)

100%|██████████| 203/203 [03:09<00:00,  1.07it/s]


Embedding matrix: (12935, 384)


In [6]:
# %%
index = faiss.IndexFlatIP(EMBED_DIM)  # cosine similarity (via normalized vectors)
index.add(X)

print("FAISS index size:", index.ntotal)

FAISS index size: 12935


In [7]:
# %%
# Save FAISS index
faiss.write_index(index, str(OUT_FAISS))

# Lookup table: faiss row → chunk_id
lookup_df = df[[
    "chunk_id",
    "pmcid",
    "article_title",
    "section",
    "license"
]].reset_index(drop=True)

lookup_df.to_parquet(OUT_LOOKUP, index=False)

# Minimal metadata (for debugging / audits)
meta = {
    "embedding_model": EMBED_MODEL,
    "embedding_dim": EMBED_DIM,
    "num_chunks": int(index.ntotal),
}

OUT_META.write_text(json.dumps(meta, indent=2))

print("Saved:")
print(" -", OUT_FAISS.resolve())
print(" -", OUT_LOOKUP.resolve())
print(" -", OUT_META.resolve())

Saved:
 - D:\Pictures\pmc_graphrag\data\index\chunks.faiss
 - D:\Pictures\pmc_graphrag\data\index\chunk_lookup.parquet
 - D:\Pictures\pmc_graphrag\data\index\chunk_meta.json


In [8]:
# %%
def retrieve_similar_chunks(
    query: str,
    top_k: int = 10,
):
    q_vec = embedder.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True,
    ).astype("float32")

    scores, idxs = index.search(q_vec, top_k)

    results = lookup_df.iloc[idxs[0]].copy()
    results["similarity"] = scores[0]

    return results.sort_values("similarity", ascending=False)

In [9]:
# %%
test_query = "fever hypotension infection confusion"

res = retrieve_similar_chunks(test_query, top_k=5)
display(res)

,chunk_id,pmcid,article_title,section,license,similarity
12230,b6eaa9d6b46d89bee4a6231d6f22881100bd8868,PMC12930594,On-scene selective brain cooling in ventricula...,results,CC BY,0.540957
917,f04581c55bdd13742e49cc3a23cdbdad20a32cf4,PMC12897580,Differences in Clinical Manifestations of Huma...,results,CC BY,0.529401
1739,94d3247fdccc70a981e218fea721b495f25fbbc1,PMC12898779,Retrospective Study of Complicated Pneumonia a...,discussion,CC BY,0.512765
874,a88246676454eb915bbc8ffcb99ca9e42ad3c5ef,PMC12897341,The Effect of Post-Transplant Cyclophosphamide...,results,CC BY,0.492082
909,fbcc5c3c161b212b1048be28a83f1893518c7460,PMC12897580,Differences in Clinical Manifestations of Huma...,discussion,CC BY,0.491996


In [10]:
# =========================
# NOTEBOOK 04 — FAISS / EMBEDDING METRICS REPORT
# =========================
import os
import json
import numpy as np
import pandas as pd

print("=" * 90)
print("NOTEBOOK 04 METRICS — EMBEDDINGS + FAISS INDEX")
print("=" * 90)

if 'df' in globals() and not df.empty:
    print(f"Input chunks: {len(df):,}")
    print(f"Input PMCIDs: {df['pmcid'].nunique():,}")
    print(f"Sections:")
    display(df["section"].value_counts(dropna=False).to_frame("count"))

if 'EMBED_MODEL' in globals():
    print(f"\nEmbedding model: {EMBED_MODEL}")

if 'EMBED_DIM' in globals():
    print(f"Embedding dimension: {EMBED_DIM}")

if 'X' in globals():
    print(f"Embedding matrix shape: {X.shape}")
    row_norms = np.linalg.norm(X, axis=1)
    print("\nEmbedding vector norm stats:")
    display(pd.Series(row_norms, name="l2_norm").describe())

if 'index' in globals():
    print(f"\nFAISS ntotal: {index.ntotal}")

if 'lookup_df' in globals() and not lookup_df.empty:
    print(f"Lookup rows: {len(lookup_df):,}")
    print(f"Lookup unique chunk_ids: {lookup_df['chunk_id'].nunique():,}")
    print(f"Lookup unique PMCIDs: {lookup_df['pmcid'].nunique():,}")

# File sizes
print("\nSaved file sizes:")
for label, path_var in [("FAISS index", "OUT_FAISS"), ("Lookup parquet", "OUT_LOOKUP"), ("Meta JSON", "OUT_META")]:
    if path_var in globals():
        p = globals()[path_var]
        if p.exists():
            size_mb = p.stat().st_size / (1024 ** 2)
            print(f"{label}: {p.resolve()}  |  {size_mb:.2f} MB")
        else:
            print(f"{label}: file not found")

# Simple retrieval sanity on fixed probes
if 'retrieve_similar_chunks' in globals():
    PROBE_QUERIES = [
        "fever hypotension infection confusion",
        "cough shortness of breath fever",
        "abdominal pain diarrhea fever",
        "chest pain hypotension dizziness",
    ]

    probe_rows = []
    for q in PROBE_QUERIES:
        try:
            res = retrieve_similar_chunks(q, top_k=5)
            probe_rows.append({
                "query": q,
                "top_similarity": float(res["similarity"].max()),
                "mean_top5_similarity": float(res["similarity"].mean()),
                "distinct_pmcids_top5": int(res["pmcid"].nunique()),
            })
        except Exception as e:
            probe_rows.append({
                "query": q,
                "top_similarity": np.nan,
                "mean_top5_similarity": np.nan,
                "distinct_pmcids_top5": np.nan,
            })

    probe_df = pd.DataFrame(probe_rows)
    print("\nProbe-query retrieval sanity:")
    display(probe_df)

if 'meta' in globals():
    print("\nMetadata JSON contents:")
    print(json.dumps(meta, indent=2))

NOTEBOOK 04 METRICS — EMBEDDINGS + FAISS INDEX
Input chunks: 12,935
Input PMCIDs: 814
Sections:


,count
section,
results,7782
discussion,5153



Embedding model: sentence-transformers/all-MiniLM-L6-v2
Embedding dimension: 384
Embedding matrix shape: (12935, 384)

Embedding vector norm stats:


count    1.293500e+04
mean     1.000000e+00
std      4.092005e-08
min      9.999999e-01
25%      1.000000e+00
50%      1.000000e+00
75%      1.000000e+00
max      1.000000e+00
Name: l2_norm, dtype: float64


FAISS ntotal: 12935
Lookup rows: 12,935
Lookup unique chunk_ids: 12,935
Lookup unique PMCIDs: 814

Saved file sizes:
FAISS index: D:\Pictures\pmc_graphrag\data\index\chunks.faiss  |  18.95 MB
Lookup parquet: D:\Pictures\pmc_graphrag\data\index\chunk_lookup.parquet  |  0.59 MB
Meta JSON: D:\Pictures\pmc_graphrag\data\index\chunk_meta.json  |  0.00 MB

Probe-query retrieval sanity:


,query,top_similarity,mean_top5_similarity,distinct_pmcids_top5
0,fever hypotension infection confusion,0.540957,0.513440,4
1,cough shortness of breath fever,0.512869,0.490899,4
2,abdominal pain diarrhea fever,0.487481,0.469440,4
3,chest pain hypotension dizziness,0.473445,0.438240,4



Metadata JSON contents:
{
  "embedding_model": "sentence-transformers/all-MiniLM-L6-v2",
  "embedding_dim": 384,
  "num_chunks": 12935
}
